In [62]:
# Import required libraries
import pandas as pd
import numpy as np
import nltk
import re

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Download NLTK data
nltk.download('stopwords')



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [63]:
from google.colab import files
uploaded = files.upload()

Saving combined_dataset.csv to combined_dataset (2).csv


In [64]:
# Load dataset
df = pd.read_csv('combined_dataset.csv', encoding='latin-1')

# Keep only required columns
df = df[['text', 'target']]
df = df.rename(columns={'text': 'messages', 'target': 'label'})

df.head()

,messages,label
0,Congratulations! You've been selected for a lu...,spam
1,URGENT: Your account has been compromised. Cli...,spam
2,You've won a free iPhone! Claim your prize by ...,spam
3,Act now and receive a 50% discount on all purc...,spam
4,Important notice: Your subscription will expir...,spam


In [65]:
# Check null values
df.isnull().sum()

,0
messages,0
label,0


In [66]:
# Text cleaning function
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^0-9a-zA-Z]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

# Apply cleaning
df['clean_text'] = df['messages'].apply(clean_text)

df.head()

,messages,label,clean_text
0,Congratulations! You've been selected for a lu...,spam,congratulations you ve been selected for a lux...
1,URGENT: Your account has been compromised. Cli...,spam,urgent your account has been compromised click...
2,You've won a free iPhone! Claim your prize by ...,spam,you ve won a free iphone claim your prize by c...
3,Act now and receive a 50% discount on all purc...,spam,act now and receive a 50 discount on all purch...
4,Important notice: Your subscription will expir...,spam,important notice your subscription will expire...


In [67]:
X = df['clean_text']
y = df['label']

In [69]:
def classify(model, X, y):
    x_train, x_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    pipeline_model = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', model)
    ])

    pipeline_model.fit(x_train, y_train)

    print("Accuracy:", pipeline_model.score(x_test, y_test) * 100)

    y_pred = pipeline_model.predict(x_test)
    print(classification_report(y_test, y_pred))

    return pipeline_model

In [70]:
model_lr = LogisticRegression()
pipeline_lr = classify(model_lr, X, y)

Accuracy: 94.41809558555272
              precision    recall  f1-score   support

         ham       0.94      0.99      0.97      2139
        spam       0.97      0.77      0.86       602

    accuracy                           0.94      2741
   macro avg       0.95      0.88      0.91      2741
weighted avg       0.95      0.94      0.94      2741



In [71]:
model_nb = MultinomialNB()
pipeline_nb = classify(model_nb, X, y)

Accuracy: 87.88763225100328
              precision    recall  f1-score   support

         ham       0.87      1.00      0.93      2139
        spam       1.00      0.45      0.62       602

    accuracy                           0.88      2741
   macro avg       0.93      0.72      0.77      2741
weighted avg       0.90      0.88      0.86      2741



In [72]:
model_svm = LinearSVC()
pipeline_svm = classify(model_svm, X, y)

Accuracy: 96.97190806275083
              precision    recall  f1-score   support

         ham       0.97      0.99      0.98      2139
        spam       0.97      0.89      0.93       602

    accuracy                           0.97      2741
   macro avg       0.97      0.94      0.95      2741
weighted avg       0.97      0.97      0.97      2741



In [73]:
model_rf = RandomForestClassifier()
pipeline_rf = classify(model_rf, X, y)

Accuracy: 94.70995986866107
              precision    recall  f1-score   support

         ham       0.95      0.99      0.97      2139
        spam       0.95      0.80      0.87       602

    accuracy                           0.95      2741
   macro avg       0.95      0.89      0.92      2741
weighted avg       0.95      0.95      0.95      2741



In [74]:
import pickle

# Save SVM model (best one)
with open('model.pkl', 'wb') as f:
    pickle.dump(pipeline_svm, f)

print("Model saved successfully!")

Model saved successfully!


In [78]:
# Load model
model = pickle.load(open('model.pkl', 'rb'))

# Test prediction
samples = [
    "Congratulations! You won a free ticket. Call now!",
    "Hey bro, let's meet tomorrow",
    "URGENT! claim your prize now",
    "Are you coming to class?"
]
for s in samples:
  clean_sample = clean_text(s)
  prediction = model.predict([clean_sample])

  print("Prediction:", prediction[0])

Prediction: spam
Prediction: ham
Prediction: spam
Prediction: ham


In [77]:
files.download('model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>